In [1]:
# Instalación de librerías necesarias
!pip install -q ucimlrepo plotly scikit-learn pandas numpy scipy

# Importación de librerías
import pandas as pd
import numpy as np
import math
import plotly.express as px
import plotly.graph_objects as go

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)
from scipy.stats import chi2_contingency

In [2]:
print("Cargando dataset desde UCI...")
congressional_voting_records = fetch_ucirepo(id=105)

# Separación de características (X) y variable objetivo (y)
X = congressional_voting_records.data.features
y = congressional_voting_records.data.targets
df = pd.concat([X, y], axis=1)

# Imputación de valores faltantes con la categoría '_??'
df.fillna('_??', inplace=True)

# Definición de columnas
features = df.columns[:-1].tolist()
target_class = 'Class'

print(f"Dataset cargado. Forma: {df.shape}")
print("Primeras filas después de imputar valores faltantes:")
display(df.head())

Cargando dataset desde UCI...
Dataset cargado. Forma: (435, 17)
Primeras filas después de imputar valores faltantes:


,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-corporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa,Class
0,n,y,n,y,y,y,n,n,n,y,_??,y,y,y,n,y,republican
1,n,y,n,y,y,y,n,n,n,n,n,y,y,y,n,_??,republican
2,_??,y,y,_??,y,y,n,n,n,n,y,n,y,y,n,n,democrat
3,n,y,y,n,_??,y,n,n,n,n,y,n,y,n,n,y,democrat
4,y,y,y,n,y,y,n,n,n,n,y,_??,y,y,y,y,democrat


In [3]:

# Variables predictoras
X = df.drop(columns=["Class"])

# One-Hot Encoding
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

X_ohe = encoder.fit_transform(X)

X_ohe = pd.DataFrame(
    X_ohe,
    columns=encoder.get_feature_names_out(X.columns)
)

print(X_ohe.shape)
X_ohe.head()


(435, 48)


,handicapped-infants__??,handicapped-infants_n,handicapped-infants_y,water-project-cost-sharing__??,water-project-cost-sharing_n,water-project-cost-sharing_y,adoption-of-the-budget-resolution__??,adoption-of-the-budget-resolution_n,adoption-of-the-budget-resolution_y,physician-fee-freeze__??,...,superfund-right-to-sue_y,crime__??,crime_n,crime_y,duty-free-exports__??,duty-free-exports_n,duty-free-exports_y,export-administration-act-south-africa__??,export-administration-act-south-africa_n,export-administration-act-south-africa_y
0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0


In [4]:
summary = X_ohe.describe().T

summary[["min","max","mean","std"]].head(20)

,min,max,mean,std
handicapped-infants__??,0.0,1.0,0.027586,0.163973
handicapped-infants_n,0.0,1.0,0.542529,0.498762
handicapped-infants_y,0.0,1.0,0.429885,0.495630
water-project-cost-sharing__??,0.0,1.0,0.110345,0.313680
water-project-cost-sharing_n,0.0,1.0,0.441379,0.497123
water-project-cost-sharing_y,0.0,1.0,0.448276,0.497890
adoption-of-the-budget-resolution__??,0.0,1.0,0.025287,0.157177
adoption-of-the-budget-resolution_n,0.0,1.0,0.393103,0.489002
adoption-of-the-budget-resolution_y,0.0,1.0,0.581609,0.493863
physician-fee-freeze__??,0.0,1.0,0.025287,0.157177


In [5]:
import plotly.express as px

stats = (
    X_ohe.mean()
    .mul(100)
    .sort_values()
    .reset_index()
)

stats.columns = ["Variable", "Porcentaje"]

stats["Variable"] = (
    stats["Variable"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
)

fig = px.bar(
    stats,
    x="Porcentaje",
    y="Variable",
    orientation="h",
    title="Frecuencia de las categorías creadas con One-HotEncoder",
    labels={
        "Porcentaje": "Observaciones con valor 1 (%)",
        "Variable": "Variable codificada"
    },
    color_discrete_sequence=["#9932CC"]
)

fig.update_layout(
    template="plotly_white",
    font=dict(color="#240A24"),
    title=dict(
        x=0.5,
        font=dict(size=21, color="#240A24")
    ),
    height=1050,
    margin=dict(l=260, r=40, t=80, b=60),
    showlegend=False
)

fig.update_xaxes(
    ticksuffix=" %",
    gridcolor="rgba(36, 10, 36, 0.12)"
)

fig.update_traces(
    marker_line_color="#240A24",
    marker_line_width=0.7,
    hovertemplate="<b>%{y}</b><br>Porcentaje: %{x:.1f} %<extra></extra>"
)

fig.show()

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_ohe)

scaled = pd.DataFrame(X_scaled)

scaled.describe().loc[["mean","std"]]

,0,1,2,3,4,5,6,7,8,9,...,38,39,40,41,42,43,44,45,46,47
mean,-8.167158e-18,-1.143402e-16,-2.450147e-17,6.533726e-17,-1.633432e-17,1.633432e-17,-2.450147e-17,-1.429253e-17,2.552237e-17,4.083579e-18,...,5.308653e-17,5.921189e-17,1.633432e-17,-1.429253e-17,1.837611e-17,-6.942084e-17,-8.575516e-17,-3.471042e-17,3.062684e-17,3.879400e-17
std,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,...,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00,1.001151e+00


In [7]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1. Seleccionar la variable que se desea analizar
# ============================================================

indice_variable = 0
nombre_columna = str(X_ohe.columns[indice_variable])

sin_escalar = (
    X_ohe.iloc[:, indice_variable]
    .astype(float)
    .to_numpy()
)

escalado = (
    np.asarray(X_scaled)[:, indice_variable]
    .astype(float)
)

# ============================================================
# 2. Identificar la variable original y la categoría One-Hot
# ============================================================

# Busca cuál de las variables originales corresponde a la columna
# generada por OneHotEncoder.
variables_coincidentes = [
    variable
    for variable in features
    if nombre_columna.startswith(variable + "_")
]

if variables_coincidentes:
    variable_original = max(
        variables_coincidentes,
        key=len
    )

    categoria = nombre_columna[
        len(variable_original) + 1:
    ]
else:
    variable_original = nombre_columna
    categoria = ""

# Nombres más claros para presentar la gráfica
nombres_legibles = {
    "handicapped-infants":
        "Apoyo a infantes con discapacidad",
    "water-project-cost-sharing":
        "Distribución de costos de proyectos de agua",
    "adoption-of-the-budget-resolution":
        "Aprobación de la resolución presupuestaria",
    "physician-fee-freeze":
        "Congelamiento de honorarios médicos",
    "el-salvador-aid":
        "Ayuda a El Salvador"
}

variable_legible = nombres_legibles.get(
    variable_original,
    variable_original.replace("-", " ").capitalize()
)

# Etiquetas según la categoría representada por la columna
if categoria in ["_??", "??"]:
    categoria_legible = "Dato faltante"
    etiqueta_cero = "Dato registrado"
    etiqueta_uno = "Dato faltante"

elif categoria.lower() == "y":
    categoria_legible = 'Respuesta "Sí"'
    etiqueta_cero = 'Otra respuesta'
    etiqueta_uno = 'Respuesta "Sí"'

elif categoria.lower() == "n":
    categoria_legible = 'Respuesta "No"'
    etiqueta_cero = 'Otra respuesta'
    etiqueta_uno = 'Respuesta "No"'

else:
    categoria_legible = categoria
    etiqueta_cero = "No pertenece"
    etiqueta_uno = "Pertenece"

# ============================================================
# 3. Calcular frecuencias y porcentajes
# ============================================================

conteo_cero = int(np.sum(sin_escalar == 0))
conteo_uno = int(np.sum(sin_escalar == 1))

total = len(sin_escalar)

porcentaje_cero = conteo_cero / total * 100
porcentaje_uno = conteo_uno / total * 100

frecuencias = [conteo_cero, conteo_uno]
porcentajes = [porcentaje_cero, porcentaje_uno]

# Valores escalados correspondientes a 0 y 1
valor_escalado_cero = float(
    np.mean(escalado[sin_escalar == 0])
)

valor_escalado_uno = float(
    np.mean(escalado[sin_escalar == 1])
)

# Etiquetas colocadas encima de las barras
texto_barras = [
    (
        f"<b>{conteo_cero}</b> observaciones"
        f"<br>{porcentaje_cero:.1f} %"
    ),
    (
        f"<b>{conteo_uno}</b> observaciones"
        f"<br>{porcentaje_uno:.1f} %"
    )
]

# ============================================================
# 4. Crear los dos paneles
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.14,
    subplot_titles=(
        "<b>Codificación One-Hot original</b>",
        "<b>Después del escalamiento</b>"
    )
)

# ============================================================
# 5. Panel izquierdo: valores 0 y 1
# ============================================================

fig.add_trace(
    go.Bar(
        x=[
            f"{etiqueta_cero}<br>(código 0)",
            f"{etiqueta_uno}<br>(código 1)"
        ],
        y=porcentajes,
        customdata=frecuencias,
        text=texto_barras,
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color="#9932CC",
            line=dict(
                color="#240A24",
                width=1.4
            )
        ),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Observaciones: %{customdata}<br>"
            "Porcentaje: %{y:.1f} %"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# ============================================================
# 6. Panel derecho: valores estandarizados
# ============================================================

fig.add_trace(
    go.Bar(
        x=[
            (
                f"{etiqueta_cero}"
                f"<br>(z = {valor_escalado_cero:.3f})"
            ),
            (
                f"{etiqueta_uno}"
                f"<br>(z = {valor_escalado_uno:.3f})"
            )
        ],
        y=porcentajes,
        customdata=frecuencias,
        text=texto_barras,
        textposition="outside",
        cliponaxis=False,
        marker=dict(
            color="#E6E6FA",
            line=dict(
                color="#240A24",
                width=1.4
            )
        ),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Observaciones: %{customdata}<br>"
            "Porcentaje: %{y:.1f} %"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ============================================================
# 7. Formato de los ejes
# ============================================================

fig.update_xaxes(
    title=dict(
        text="<b>Estado de la observación</b>",
        standoff=20
    ),
    tickfont=dict(size=13),
    showline=True,
    linewidth=1.2,
    linecolor="#240A24",
    mirror=True,
    ticks="outside",
    tickcolor="#240A24",
    row=1,
    col=1
)

fig.update_xaxes(
    title=dict(
        text="<b>Estado de la observación</b>",
        standoff=20
    ),
    tickfont=dict(size=13),
    showline=True,
    linewidth=1.2,
    linecolor="#240A24",
    mirror=True,
    ticks="outside",
    tickcolor="#240A24",
    row=1,
    col=2
)

fig.update_yaxes(
    title=dict(
        text="<b>Porcentaje de observaciones (%)</b>",
        standoff=12
    ),
    range=[0, 108],
    dtick=20,
    ticksuffix=" %",
    showline=True,
    linewidth=1.2,
    linecolor="#240A24",
    mirror=True,
    ticks="outside",
    tickcolor="#240A24",
    gridcolor="rgba(36, 10, 36, 0.13)",
    zeroline=False,
    row=1,
    col=1
)

fig.update_yaxes(
    range=[0, 108],
    dtick=20,
    ticksuffix=" %",
    showline=True,
    linewidth=1.2,
    linecolor="#240A24",
    mirror=True,
    ticks="outside",
    tickcolor="#240A24",
    gridcolor="rgba(36, 10, 36, 0.13)",
    zeroline=False,
    row=1,
    col=2
)

# ============================================================
# 8. Título y formato general
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "<b>Distribución de una variable One-Hot antes y "
            "después del escalamiento</b>"
            f"<br><sup>Variable: {variable_legible} | "
            f"Categoría representada: {categoria_legible}</sup>"
            "<br><sup>El escalamiento modifica los valores "
            "numéricos, pero mantiene las mismas frecuencias.</sup>"
        ),
        x=0.5,
        xanchor="center",
        y=0.97,
        font=dict(
            family="Georgia",
            size=22,
            color="#240A24"
        )
    ),
    template="plotly_white",
    font=dict(
        family="Georgia",
        size=14,
        color="#240A24"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    bargap=0.38,
    height=650,
    width=1100,
    margin=dict(
        l=100,
        r=50,
        t=155,
        b=120
    ),
    uniformtext=dict(
        minsize=11,
        mode="show"
    )
)

fig.show()

In [8]:
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score

scores=[]

for k in range(2,11):

    model=KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels=model.fit_predict(X_ohe)

    scores.append(
        calinski_harabasz_score(
            X_ohe,
            labels
        )
    )

In [9]:
scores_scaled=[]

for k in range(2,11):

    model=KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels=model.fit_predict(X_scaled)

    scores_scaled.append(
        calinski_harabasz_score(
            X_scaled,
            labels
        )
    )

In [10]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(2,11)),
    y=scores,
    mode="lines+markers",
    name="Sin escalamiento",
    line=dict(color="#9932CC", width=3)
))

fig.add_trace(go.Scatter(
    x=list(range(2,11)),
    y=scores_scaled,
    mode="lines+markers",
    name="Con StandardScaler",
    line=dict(color="#E6E6FA", width=3)
))

fig.update_layout(
    template="plotly_white",
    title="Comparación del índice Calinski-Harabasz",
    title_font_color="#240A24",
    font_color="#240A24"
)

fig.show()

In [11]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_ohe)

In [12]:
tabla = pd.crosstab(df["Class"], df["Cluster"])
print(tabla)

Cluster       0    1
Class               
democrat     42  225
republican  158   10


In [13]:
chi2, p, dof, expected = chi2_contingency(tabla)

phi2 = chi2 / len(df)

cramers_v = math.sqrt(
    phi2 /
    min(tabla.shape[0]-1, tabla.shape[1]-1)
)

tschuprows_t = math.sqrt(
    phi2 /
    (
        (tabla.shape[0]-1) *
        (tabla.shape[1]-1)
    )
)

print(f"Chi2: {chi2:.2f}")
print(f"p-value: {p:.6f}")
print(f"Phi²: {phi2:.4f}")
print(f"Cramer's V: {cramers_v:.4f}")
print(f"Tschuprow's T: {tschuprows_t:.4f}")

Chi2: 251.50
p-value: 0.000000
Phi²: 0.5782
Cramer's V: 0.7604
Tschuprow's T: 0.7604


In [14]:
pd.crosstab(df["Class"], df["Cluster"])

Cluster,0,1
Class,,
democrat,42,225
republican,158,10


In [15]:
df["Predicted"] = df["Cluster"].replace({
    0: "republican",
    1: "democrat"
})

In [16]:
from IPython.display import display, HTML
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Definir cuál clase se considera positiva
clase_positiva = "democrat"
clase_negativa = "republican"

# Matriz de confusión
cm = confusion_matrix(
    df["Class"],
    df["Predicted"],
    labels=[clase_negativa, clase_positiva]
)

vn, fp, fn, vp = cm.ravel()

# Tabla con la identificación de cada resultado
tabla_cm = pd.DataFrame(
    [
        [f"{vn} (VN)", f"{fp} (FP)"],
        [f"{fn} (FN)", f"{vp} (VP)"]
    ],
    index=[
        f"Real: {clase_negativa}",
        f"Real: {clase_positiva}"
    ],
    columns=[
        f"Predicho: {clase_negativa}",
        f"Predicho: {clase_positiva}"
    ]
)

display(tabla_cm)

# Espacio después de la tabla
display(HTML("<div style='height: 40px;'></div>"))

print(
    classification_report(
        df["Class"],
        df["Predicted"]
    )
)

print(
    "Accuracy:",
    accuracy_score(
        df["Class"],
        df["Predicted"]
    )
)

,Predicho: republican,Predicho: democrat
Real: republican,158 (VN),10 (FP)
Real: democrat,42 (FN),225 (VP)


              precision    recall  f1-score   support

    democrat       0.96      0.84      0.90       267
  republican       0.79      0.94      0.86       168

    accuracy                           0.88       435
   macro avg       0.87      0.89      0.88       435
weighted avg       0.89      0.88      0.88       435

Accuracy: 0.8804597701149425
